In [106]:
from pprint import pprint

import numpy as np
import geopandas as gpd

import folium
import networkx as nx
from folium import PolyLine, CircleMarker

import os
import sys

parent_dir = os.path.abspath("..")
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from utils.load_data import load_processed_original_data, resample_data  # noqa: E402
from utils.constants import FIGURES_DIR, RAW_DATA_DIR  # noqa: E402

MAP_FIGURES_DIR = FIGURES_DIR / "maps"
MAP_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LOCALIDAD_SHAPEFILE = RAW_DATA_DIR / "poligonos-localidades.zip"

In [44]:
# Load the processed original data as a GeoDataFrame
original_gdf = load_processed_original_data(as_geopandas=True, parse_dates=True)

localities_cols = list(original_gdf["LOCALIDAD"].unique())

# Load the locality shapefile
localities = gpd.read_file(LOCALIDAD_SHAPEFILE).to_crs("EPSG:4326")
localities["centroid"] = localities["geometry"].centroid.to_crs("EPSG:4326")

# Filter localities to only those present in the original data
localities = localities[localities["Nombre_de_l"].isin(localities_cols)]

# Drop unnecesary columns and rename
localities.drop(columns=["Acto_admini", "Area_de_la_", "Identificad"], inplace=True)
localities.rename(columns={"Nombre_de_l": "LOCALIDAD"}, inplace=True)

localities_centroids = (
    localities[["LOCALIDAD", "centroid"]].set_index("LOCALIDAD").to_dict()["centroid"]
)

localities_centroids = dict(
    map(
        lambda item: (item[0], (item[1].xy[0][0], item[1].xy[1][0])),
        localities_centroids.items(),
    )
)

display(localities, localities_centroids)

/tmp/ipykernel_80167/1429013853.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  localities["centroid"] = localities["geometry"].centroid.to_crs("EPSG:4326")


,LOCALIDAD,geometry,centroid
0,SANTA FE,"POLYGON ((-73.99446 4.61425, -73.99446 4.61425...",POINT (-74.03622 4.59397)
1,BARRIOS UNIDOS,"POLYGON ((-74.05725 4.68684, -74.06249 4.65594...",POINT (-74.07355 4.66957)
2,FONTIBON,"POLYGON ((-74.10342 4.65351, -74.1075 4.64823,...",POINT (-74.14015 4.67818)
3,ENGATIVA,"POLYGON ((-74.15547 4.71798, -74.15547 4.71798...",POINT (-74.11319 4.70113)
4,CANDELARIA,"POLYGON ((-74.06621 4.60317, -74.0662 4.60317,...",POINT (-74.07207 4.59661)
5,TUNJUELITO,"POLYGON ((-74.13777 4.59489, -74.13165 4.59363...",POINT (-74.13599 4.57484)
6,PUENTE ARANDA,"POLYGON ((-74.1183 4.63741, -74.11504 4.64053,...",POINT (-74.11158 4.61625)
7,RAFAEL URIBE URIBE,"POLYGON ((-74.12803 4.59254, -74.12777 4.59233...",POINT (-74.11336 4.56648)
8,KENNEDY,"POLYGON ((-74.1183 4.63741, -74.11845 4.63727,...",POINT (-74.15267 4.63035)
9,USME,"POLYGON ((-74.05597 4.50832, -74.05611 4.50822...",POINT (-74.14281 4.39003)


{'SANTA FE': (-74.03621758151407, 4.593972728724646),
 'BARRIOS UNIDOS': (-74.07355160510069, 4.669566933458008),
 'FONTIBON': (-74.14015130074621, 4.67818292595219),
 'ENGATIVA': (-74.11318770329797, 4.701125563272906),
 'CANDELARIA': (-74.07207079532778, 4.596605075136357),
 'TUNJUELITO': (-74.13598945142, 4.574841398493438),
 'PUENTE ARANDA': (-74.11158023352897, 4.616245470318943),
 'RAFAEL URIBE URIBE': (-74.1133631829354, 4.56647690086027),
 'KENNEDY': (-74.1526676238341, 4.630347351356093),
 'USME': (-74.14280649581963, 4.390025128368835),
 'LOS MARTIRES': (-74.08794786812676, 4.607155934954452),
 'CIUDAD BOLIVAR': (-74.1619589044063, 4.482460265935735),
 'SUBA': (-74.07584527554698, 4.763208121309923),
 'BOSA': (-74.19438893446517, 4.621779959835892),
 'ANTONIO NARIÑO': (-74.1028424006145, 4.588770674796082),
 'CHAPINERO': (-74.03688302787356, 4.644982720078631),
 'SAN CRISTOBAL': (-74.0660725605067, 4.548749008988842),
 'TEUSAQUILLO': (-74.08576901958142, 4.641173586592106),
 

## Create graph

Create a graph using the distance matrix (obtained from correlation matrix) as adjecency matrix for our graph. Then, extract the MST (Minimum Spanning Tree) to get the localities with least distance (positive linear correlation)

In [45]:
df = resample_data(freq="1W", multi_index=True)

# Extraer las columnas de localidades
localidad_cols = list(df["LOCALIDAD"].columns)

# Create correlation matrix between LOCALIDAD columns
corr_matrix = df["LOCALIDAD"][localidad_cols].corr()

dist_matrix = np.sqrt(2 * (1 - corr_matrix))  # Euclidean distance from correlation

dist_matrix

,KENNEDY,ENGATIVA,FONTIBON,SUBA,PUENTE ARANDA,CHAPINERO,TEUSAQUILLO,USAQUEN,LOS MARTIRES,SANTA FE,BARRIOS UNIDOS,RAFAEL URIBE URIBE,BOSA,SAN CRISTOBAL,TUNJUELITO,CIUDAD BOLIVAR,ANTONIO NARIÑO,USME,CANDELARIA
KENNEDY,0.000000,1.030803,0.983447,1.048664,1.013150,1.076638,1.110893,1.110359,1.371934,1.253834,1.130930,1.056088,1.011972,1.058836,1.031440,0.877456,1.215732,1.117121,1.350635
ENGATIVA,1.030803,0.000000,1.048863,1.034521,1.091270,1.091195,1.098128,1.098972,1.236289,1.190277,1.117987,1.216537,1.173354,1.304352,1.231388,1.127200,1.244523,1.294174,1.297206
FONTIBON,0.983447,1.048863,0.000000,1.061083,1.126262,1.135735,1.132245,1.113440,1.319530,1.245123,1.134424,1.213050,1.110525,1.258607,1.156900,1.036247,1.230011,1.225881,1.336940
SUBA,1.048664,1.034521,1.061083,0.000000,1.125208,1.143038,1.127286,1.107581,1.252651,1.224023,1.142502,1.218116,1.160878,1.279303,1.228227,1.070888,1.261765,1.302934,1.311473
PUENTE ARANDA,1.013150,1.091270,1.126262,1.125208,0.000000,1.020273,1.086142,1.057365,1.225136,1.206984,1.042773,1.210793,1.205620,1.249856,1.241407,1.173750,1.272709,1.333129,1.247623
CHAPINERO,1.076638,1.091195,1.135735,1.143038,1.020273,0.000000,0.902596,0.870659,1.139991,0.948690,0.932924,1.260819,1.332885,1.209040,1.276509,1.301095,1.254248,1.361848,1.163694
TEUSAQUILLO,1.110893,1.098128,1.132245,1.127286,1.086142,0.902596,0.000000,0.949339,1.181125,1.034800,0.971768,1.276675,1.343061,1.229523,1.281592,1.281191,1.276751,1.357566,1.195375
USAQUEN,1.110359,1.098972,1.113440,1.107581,1.057365,0.870659,0.949339,0.000000,1.159340,1.086919,0.985095,1.280324,1.350449,1.230441,1.245920,1.251884,1.283323,1.339167,1.209071
LOS MARTIRES,1.371934,1.236289,1.319530,1.252651,1.225136,1.139991,1.181125,1.159340,0.000000,1.106612,1.153454,1.428487,1.480064,1.412731,1.473727,1.496758,1.314175,1.475854,1.180392
SANTA FE,1.253834,1.190277,1.245123,1.224023,1.206984,0.948690,1.034800,1.086919,1.106612,0.000000,1.079688,1.376003,1.450350,1.307023,1.351587,1.421041,1.346523,1.421857,1.210327


In [28]:
graph = nx.from_pandas_adjacency(dist_matrix)

In [74]:
mst = nx.minimum_spanning_tree(graph, algorithm="prim")

In [75]:
mst_df = nx.to_pandas_adjacency(mst)
mst_df

,KENNEDY,ENGATIVA,FONTIBON,SUBA,PUENTE ARANDA,CHAPINERO,TEUSAQUILLO,USAQUEN,LOS MARTIRES,SANTA FE,BARRIOS UNIDOS,RAFAEL URIBE URIBE,BOSA,SAN CRISTOBAL,TUNJUELITO,CIUDAD BOLIVAR,ANTONIO NARIÑO,USME,CANDELARIA
KENNEDY,0.000000,1.030803,0.983447,0.000000,1.013150,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.056088,0.000000,1.058836,0.000000,0.877456,1.215732,0.000000,0.000000
ENGATIVA,1.030803,0.000000,0.000000,1.034521,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
FONTIBON,0.983447,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
SUBA,0.000000,1.034521,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PUENTE ARANDA,1.013150,0.000000,0.000000,0.000000,0.000000,1.020273,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
CHAPINERO,0.000000,0.000000,0.000000,0.000000,1.020273,0.000000,0.902596,0.870659,0.000000,0.948690,0.932924,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.163694
TEUSAQUILLO,0.000000,0.000000,0.000000,0.000000,0.000000,0.902596,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
USAQUEN,0.000000,0.000000,0.000000,0.000000,0.000000,0.870659,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LOS MARTIRES,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.106612,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
SANTA FE,0.000000,0.000000,0.000000,0.000000,0.000000,0.948690,0.000000,0.000000,1.106612,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Metrics

With the MST it's possible to get some interesting metrics for each node (locality) to rank their importance, such as:

* Degree Centrality
* Betweenness Centrality
* Closeness Centrality

In [104]:
def sort_centrality(
    centrality: dict[str, int | float],
    reverse: bool = True,
) -> list[tuple[str, int | float]]:
    return sorted(centrality.items(), key=lambda item: item[1], reverse=reverse)

In [97]:
degree_centrality = nx.degree_centrality(mst)
betweenness_centrality = nx.betweenness_centrality(mst)
closeness_centrality = nx.closeness_centrality(mst)

In [103]:
degree_centrality

{'KENNEDY': 0.38888888888888884,
 'ENGATIVA': 0.1111111111111111,
 'FONTIBON': 0.05555555555555555,
 'SUBA': 0.05555555555555555,
 'PUENTE ARANDA': 0.1111111111111111,
 'CHAPINERO': 0.3333333333333333,
 'TEUSAQUILLO': 0.05555555555555555,
 'USAQUEN': 0.05555555555555555,
 'LOS MARTIRES': 0.05555555555555555,
 'SANTA FE': 0.1111111111111111,
 'BARRIOS UNIDOS': 0.05555555555555555,
 'RAFAEL URIBE URIBE': 0.05555555555555555,
 'BOSA': 0.05555555555555555,
 'SAN CRISTOBAL': 0.05555555555555555,
 'TUNJUELITO': 0.05555555555555555,
 'CIUDAD BOLIVAR': 0.2222222222222222,
 'ANTONIO NARIÑO': 0.05555555555555555,
 'USME': 0.05555555555555555,
 'CANDELARIA': 0.05555555555555555}

In [105]:
TOP = 5
top = min(TOP, len(mst.nodes))

print(f"Degree Centrality (sorted) - {top}:")
pprint(sort_centrality(degree_centrality)[:top])

print(f"\nBetweenness Centrality (sorted) - {top}:")
pprint(sort_centrality(betweenness_centrality)[:top])

print(f"\nCloseness Centrality (sorted) - {top}:")
pprint(sort_centrality(closeness_centrality)[:top])

Degree Centrality (sorted) - 5:
[('KENNEDY', 0.38888888888888884),
 ('CHAPINERO', 0.3333333333333333),
 ('CIUDAD BOLIVAR', 0.2222222222222222),
 ('ENGATIVA', 0.1111111111111111),
 ('PUENTE ARANDA', 0.1111111111111111)]

Betweenness Centrality (sorted) - 5:
[('KENNEDY', 0.7712418300653595),
 ('CHAPINERO', 0.5620915032679739),
 ('PUENTE ARANDA', 0.5032679738562091),
 ('CIUDAD BOLIVAR', 0.3137254901960784),
 ('ENGATIVA', 0.11111111111111112)]

Closeness Centrality (sorted) - 5:
[('KENNEDY', 0.5),
 ('PUENTE ARANDA', 0.46153846153846156),
 ('CHAPINERO', 0.4090909090909091),
 ('CIUDAD BOLIVAR', 0.3829787234042553),
 ('ENGATIVA', 0.35294117647058826)]


## Plot graph

In [56]:
folium_colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 
                'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue',
                'darkpurple', 'pink', 'lightblue', 'lightgreen',
                'gray', 'black', 'lightgray']

In [79]:
m = folium.Map(
    location=[4.6485784, -74.1031911],
    zoom_start=11,
    tiles="openstreetmap",
)

# Add locality polygons to the map
geo_df_list = [[point.xy[1][0], point.xy[0][0]] for point in localities.centroid]
n_localities = localities.shape[0]
locality_color = dict(
    (loc, folium_colors[i % len(folium_colors)])
    for i, loc in enumerate(localities.LOCALIDAD)
)

for i, coordinates in enumerate(geo_df_list):
    # Place the markers with the popup labels and data
    locality = localities["LOCALIDAD"].iloc[i]

    # Plot the locality polygons
    polygon = folium.vector_layers.Polygon(
        locations=[
            [coord[1], coord[0]]
            for coord in localities["geometry"].iloc[i].exterior.coords
        ],
        color=locality_color[locality],
        fill=True,
        fill_color=locality_color[locality],
        fill_opacity=0.2,
    )
    polygon.add_to(m)

# Add edges from the MST to the map
for edge in mst.edges():
    node1, node2 = edge
    coord1 = localities_centroids[node1]
    coord2 = localities_centroids[node2]

    # Obtener el peso de la arista (distancia)
    weight = mst[node1][node2].get("weight", dist_matrix.loc[node1, node2])

    # Draw the edge between the two nodes
    PolyLine(
        locations=[(coord1[1], coord1[0]), (coord2[1], coord2[0])],
        color="blue",
        weight=2,
        opacity=0.8,
        tooltip=f"{node1} - {node2}: {weight:.2f}",
    ).add_to(m)

# Add node as markers to the map
for node, coord in localities_centroids.items():
    CircleMarker(
        location=[coord[1], coord[0]],
        radius=8,
        popup=(
            f"{node}\n"
            f"Degree: {degree_centrality[node]:.2f}\n"
            f"Betweenness: {betweenness_centrality[node]:.2f}\n"
            f"Closeness: {closeness_centrality[node]:.2f}"
        ),
        color="red",
        fill=True,
        fillColor="red",
        fillOpacity=0.9,
    ).add_to(m)

m.save(MAP_FIGURES_DIR / "localities_graph.html")

/tmp/ipykernel_80167/1223489584.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  geo_df_list = [[point.xy[1][0], point.xy[0][0]] for point in localities.centroid]
